#  Big Data con PySpark — Notebook 6
## Pipeline Completo de Análisis: Preguntas de Negocio

Este notebook responde **5 preguntas reales de negocio** usando todo lo aprendido.

---

**Preguntas a responder:**
1. ¿Qué aerolinea tiene la mejor puntualidad por ruta?
2. ¿Cuáles son los horarios con mayor retraso?
3. ¿Qué rutas generan más ingreso por pasajero?
4. ¿Cómo evolucionó la demanda mensualmente?
5. ¿Qué perfil tiene un vuelo de alto retraso?

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import IntegerType

spark = (
    SparkSession.builder
    .appName("Vuelos_Pipeline_Final")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

df = spark.read.parquet("/content/drive/MyDrive/Colab Notebooks/vuelos_limpio.parquet")
df.cache()
print(f"Dataset cargado: {df.count():,} vuelos")

Dataset cargado: 446,399 vuelos


---
## Pregunta 1: ¿Qué aerolinea tiene la mejor puntualidad por ruta?

Queremos saber, para las rutas más frecuentes, qué aerolinea tiene más vuelos a tiempo.

In [5]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Paso 1: Crear columnas derivadas y calcular métricas por aerolínea y ruta
# (Se define puntualidad como retraso_min <= 15 min; ajusta a <= 0 si tu métrica es estricta)
puntualidad = (
    df
    .filter(F.col("estado") != "CANCELADO")
    .withColumn("ruta", F.concat_ws("-", F.col("origen"), F.col("destino")))
    .withColumn("es_puntual", F.when(F.col("retraso_min") <= 15, 1).otherwise(0))
    .groupBy("ruta", "aerolinea")
    .agg(
        F.count("*").alias("vuelos"),
        F.round(F.avg("es_puntual") * 100, 1).alias("pct_puntual"),
        F.round(F.avg("retraso_min"), 1).alias("retraso_prom")
    )
    .filter(F.col("vuelos") >= 500)
)

# Paso 2: Rankear aerolíneas dentro de cada ruta por puntualidad
ventana_ruta = Window.partitionBy("ruta").orderBy(F.desc("pct_puntual"))

mejor_aerolinea_por_ruta = (
    puntualidad
    .withColumn("rank", F.rank().over(ventana_ruta))
    .filter(F.col("rank") == 1)
    .orderBy(F.desc("pct_puntual"))
    .select("ruta", "aerolinea", "vuelos", "pct_puntual", "retraso_prom")
)

print("Mejor aerolinea por puntualidad en cada ruta:")
mejor_aerolinea_por_ruta.show(15, truncate=False)

Mejor aerolinea por puntualidad en cada ruta:
+-------+---------+------+-----------+------------+
|ruta   |aerolinea|vuelos|pct_puntual|retraso_prom|
+-------+---------+------+-----------+------------+
|BOG-CLO|EasyFly  |501   |77.0       |30.7        |
|VVC-CLO|Wingo    |722   |76.6       |29.2        |
|MTR-BOG|Avianca  |1530  |75.8       |31.0        |
|SMR-MDE|EasyFly  |516   |75.8       |32.3        |
|CLO-MTR|Wingo    |675   |75.6       |31.4        |
|LET-MDE|Wingo    |676   |75.3       |31.0        |
|CTG-MDE|Wingo    |730   |75.1       |33.5        |
|CLO-SMR|LATAM    |1126  |75.0       |33.5        |
|PEI-VVC|Wingo    |653   |74.7       |33.5        |
|CTG-MTR|Wingo    |739   |74.6       |31.6        |
|CLO-BAQ|Avianca  |1587  |74.6       |31.9        |
|BOG-SMR|Wingo    |753   |74.6       |33.6        |
|MTR-SMR|Wingo    |677   |74.6       |33.7        |
|VVC-LET|Wingo    |685   |74.6       |33.1        |
|CTG-VVC|LATAM    |1131  |74.5       |31.9        |
+-------+---------

In [6]:
# ¿Qué aerolinea gana más rutas?
(
    mejor_aerolinea_por_ruta
    .groupBy("aerolinea")
    .agg(F.count("*").alias("rutas_ganadas"))
    .orderBy(F.desc("rutas_ganadas"))
    .show()
)

+---------+-------------+
|aerolinea|rutas_ganadas|
+---------+-------------+
|    Wingo|           31|
|    LATAM|           30|
|  Avianca|           29|
|  EasyFly|            2|
|   Satena|            1|
+---------+-------------+



---
##  Pregunta 2: ¿Cuáles son los horarios con mayor retraso?

In [7]:
retraso_por_hora = (
    df
    .filter(F.col("retraso_min") > 0)       # solo vuelos que sí tuvieron retraso
    .groupBy("hora")
    .agg(
        F.count("*").alias("vuelos_demorados"),
        F.round(F.avg("retraso_min"), 1).alias("retraso_prom_min"),
        F.round(F.max("retraso_min"), 0).alias("retraso_max_min")
    )
    .orderBy("hora")
)

# Calcular el % de vuelos demorados por hora respecto al total de esa hora
total_por_hora = df.groupBy("hora").agg(F.count("*").alias("total_vuelos"))

analisis_horario = (
    retraso_por_hora
    .join(F.broadcast(total_por_hora), on="hora", how="inner")
    # on="hora" → join por columna con el mismo nombre en ambos DFs
    .withColumn(
        "pct_demorados",
        F.round(F.col("vuelos_demorados") / F.col("total_vuelos") * 100, 1)
    )
    .orderBy(F.desc("retraso_prom_min"))
    .select("hora", "total_vuelos", "vuelos_demorados", "pct_demorados", "retraso_prom_min")
)

print("⏰ Horas del día con mayor retraso promedio:")
analisis_horario.show(24)

⏰ Horas del día con mayor retraso promedio:
+----+------------+----------------+-------------+----------------+
|hora|total_vuelos|vuelos_demorados|pct_demorados|retraso_prom_min|
+----+------------+----------------+-------------+----------------+
|  23|       18507|            5592|         30.2|           122.2|
|   3|       18551|            5473|         29.5|           121.8|
|  13|       18344|            5614|         30.6|           121.4|
|  11|       18726|            5497|         29.4|           121.3|
|   1|       18672|            5609|         30.0|           121.2|
|   8|       18701|            5566|         29.8|           121.2|
|  21|       18552|            5499|         29.6|           121.0|
|  15|       18723|            5608|         30.0|           120.8|
|   5|       18617|            5476|         29.4|           120.6|
|  12|       18648|            5664|         30.4|           120.5|
|   6|       18459|            5548|         30.1|           120.5|
|   

---
##  Pregunta 3: ¿Qué rutas generan más ingreso por pasajero?

In [9]:
from pyspark.sql import functions as F

# 1. Preparar columnas calculadas y limpiar cancelados
df_preparado = (
    df
    .filter(F.col("estado") != "CANCELADO")
    .withColumn("ruta", F.concat_ws("-", F.col("origen"), F.col("destino")))
    .withColumn("ingreso_total_usd", F.col("pasajeros") * F.col("tarifa_usd"))
    # Si no tienes 'tipo_ruta', puedes clasificarla por distancia (ajusta los umbrales si lo requieres):
    .withColumn(
        "tipo_ruta",
        F.when(F.col("distancia_km") < 800, "CORTA")
        .when(F.col("distancia_km") < 2000, "MEDIA")
        .otherwise("LARGA")
    )
)

# 2. Agrupación y cálculo de ingreso por pasajero
ingreso_por_pasajero = (
    df_preparado
    .groupBy("ruta", "tipo_ruta")
    .agg(
        F.count("*").alias("vuelos"),
        F.round(F.sum("ingreso_total_usd"), 0).alias("ingreso_total"),
        F.round(F.sum("pasajeros"), 0).alias("pasajeros_total"),
        F.round(F.avg("distancia_km"), 0).alias("distancia_prom")
    )
    .withColumn(
        "ingreso_por_pasajero",
        F.round(F.col("ingreso_total") / F.col("pasajeros_total"), 2)
    )
    .filter(F.col("vuelos") >= 300)
    .orderBy(F.desc("ingreso_por_pasajero"))
)

print("Top 15 rutas por ingreso por pasajero:")
ingreso_por_pasajero.show(15, truncate=False)

Top 15 rutas por ingreso por pasajero:
+-------+---------+------+-------------+---------------+--------------+--------------------+
|ruta   |tipo_ruta|vuelos|ingreso_total|pasajeros_total|distancia_prom|ingreso_por_pasajero|
+-------+---------+------+-------------+---------------+--------------+--------------------+
|BOG-LET|CORTA    |1367  |7.0543881E7  |157441         |426.0         |448.07              |
|LET-CTG|LARGA    |961   |4.8663951E7  |109777         |2243.0        |443.3               |
|CLO-MDE|LARGA    |987   |5.0020769E7  |112937         |2251.0        |442.91              |
|VVC-LET|LARGA    |934   |4.7880794E7  |108540         |2251.0        |441.14              |
|PEI-BAQ|LARGA    |977   |4.9304456E7  |111948         |2244.0        |440.42              |
|MDE-LET|LARGA    |932   |4.7337252E7  |107748         |2248.0        |439.33              |
|CLO-CTG|CORTA    |1360  |6.9041468E7  |157225         |435.0         |439.13              |
|MTR-VVC|LARGA    |980   |4.922

---
##  Pregunta 4: ¿Cómo evolucionó la demanda mensualmente?

In [12]:
evolucion = (
    df
    .groupBy("anio", "mes")
    .agg(
        F.count("*").alias("vuelos"),
        F.sum("pasajeros").alias("pasajeros"),
        F.round(F.avg("tarifa_usd"), 2).alias("tarifa_prom")
    )
    .orderBy("anio", "mes")
)

# Añadir variación mes a mes (Month-over-Month)
ventana_tiempo = Window.orderBy("anio", "mes")

evolucion_con_mom = (
    evolucion
    .withColumn(
        "vuelos_mes_ant",
        F.lag("vuelos", 1).over(ventana_tiempo)  # vuelos del mes anterior
    )
    .withColumn(
        "variacion_mom_pct",
        F.round(
            (F.col("vuelos") - F.col("vuelos_mes_ant")) / F.col("vuelos_mes_ant") * 100,
            1
        )
    )
    .select("anio", "mes", "vuelos", "pasajeros", "tarifa_prom", "variacion_mom_pct")
)

print("📈 Evolución mensual de vuelos y pasajeros:")
evolucion_con_mom.show(30)


Tasa de alto retraso (>=90 min) por aerolinea:
+---------+------+----------------+------------+
|aerolinea|vuelos|pct_alto_retraso|retraso_prom|
+---------+------+----------------+------------+
|LATAM    |104027|18.84           |36.1        |
|EasyFly  |41412 |18.79           |35.9        |
|Avianca  |144972|18.74           |35.8        |
|Wingo    |62299 |18.62           |35.6        |
|Satena   |41369 |18.54           |35.5        |
|JetBlue  |20953 |18.45           |35.3        |
+---------+------+----------------+------------+


Tasa de alto retraso (>=90 min) por clase:
+---------+------+----------------+------------+
|clase    |vuelos|pct_alto_retraso|retraso_prom|
+---------+------+----------------+------------+
|Economica|311890|18.77           |35.9        |
|Primera  |20504 |18.68           |35.9        |
|Business |82638 |18.53           |35.5        |
+---------+------+----------------+------------+


Tasa de alto retraso (>=90 min) por tipo_ruta:
+---------+------+-------

---
##  Pregunta 5: ¿Qué perfil tiene un vuelo de alto retraso?

In [14]:
from pyspark.sql import functions as F

umbral = 90

# 1. Crear variables derivadas y calcular la bandera de alto retraso
df_perfil = (
    df
    .filter(F.col("estado") != "CANCELADO")
    .withColumn(
        "alto_retraso",
        F.when(F.col("retraso_min") >= umbral, 1).otherwise(0)
    )
    .withColumn(
        "tipo_ruta",
        F.when(F.col("distancia_km") < 800, "CORTA")
        .when(F.col("distancia_km") < 2000, "MEDIA")
        .otherwise("LARGA")
    )
    .withColumn(
        "franja_dia",
        F.when((F.col("hora") >= 0) & (F.col("hora") < 6), "MADRUGADA")
        .when((F.col("hora") >= 6) & (F.col("hora") < 12), "MAÑANA")
        .when((F.col("hora") >= 12) & (F.col("hora") < 18), "TARDE")
        .otherwise("NOCHE")
    )
)

# 2. Tasa de alto retraso por variable
for col_analisis in ["aerolinea", "clase", "tipo_ruta", "franja_dia"]:
    print(f"\n{'='*50}")
    print(f"Tasa de alto retraso (>={umbral} min) por {col_analisis}:")
    (
        df_perfil
        .groupBy(col_analisis)
        .agg(
            F.count("*").alias("vuelos"),
            F.round(F.avg("alto_retraso") * 100, 2).alias("pct_alto_retraso"),
            F.round(F.avg("retraso_min"), 1).alias("retraso_prom")
        )
        .orderBy(F.desc("pct_alto_retraso"))
        .show(truncate=False)
    )


Tasa de alto retraso (>=90 min) por aerolinea:
+---------+------+----------------+------------+
|aerolinea|vuelos|pct_alto_retraso|retraso_prom|
+---------+------+----------------+------------+
|LATAM    |104027|18.84           |36.1        |
|EasyFly  |41412 |18.79           |35.9        |
|Avianca  |144972|18.74           |35.8        |
|Wingo    |62299 |18.62           |35.6        |
|Satena   |41369 |18.54           |35.5        |
|JetBlue  |20953 |18.45           |35.3        |
+---------+------+----------------+------------+


Tasa de alto retraso (>=90 min) por clase:
+---------+------+----------------+------------+
|clase    |vuelos|pct_alto_retraso|retraso_prom|
+---------+------+----------------+------------+
|Economica|311890|18.77           |35.9        |
|Primera  |20504 |18.68           |35.9        |
|Business |82638 |18.53           |35.5        |
+---------+------+----------------+------------+


Tasa de alto retraso (>=90 min) por tipo_ruta:
+---------+------+-------

---
##  Exportar resultados

In [15]:
# Guardar los reportes como CSV para presentación
reportes = {
    "puntualidad_por_ruta": mejor_aerolinea_por_ruta,
    "retraso_por_hora":     analisis_horario,
    "ingreso_por_pasajero": ingreso_por_pasajero,
    "evolucion_mensual":    evolucion_con_mom,
}

for nombre, resultado_df in reportes.items():
    ruta_salida = f"reporte_{nombre}"
    (
        resultado_df
        .coalesce(1)             # reduce a 1 sola partición → 1 solo archivo CSV
        # Sin coalesce, Spark escribe 1 archivo por partición (ej: 8 archivos)
        .write
        .mode("overwrite")
        .option("header", "true")
        .csv(ruta_salida)
    )
    print(f" {nombre} → {ruta_salida}")

 puntualidad_por_ruta → reporte_puntualidad_por_ruta
 retraso_por_hora → reporte_retraso_por_hora
 ingreso_por_pasajero → reporte_ingreso_por_pasajero
 evolucion_mensual → reporte_evolucion_mensual


In [16]:
# Liberar recursos
df.unpersist()    # liberar caché del DataFrame principal
spark.stop()      # cerrar la sesión de Spark
print(" Sesión Spark cerrada")

 Sesión Spark cerrada


---
##  Resumen completo del curso

```
NOTEBOOK 1 — Carga y exploración
  spark.read.option(...).csv()     → leer CSV
  df.printSchema()                 → ver tipos
  df.describe()                    → estadísticas descriptivas
  df.cache()                       → guardar en memoria

NOTEBOOK 2 — Filtros y limpieza
  df.select() / df.drop()          → seleccionar columnas
  df.filter() / df.where()         → filtrar filas
  df.dropna() / df.fillna()        → manejar nulos
  df.dropDuplicates()              → eliminar duplicados
  col.cast() / to_timestamp()      → cambiar tipos

NOTEBOOK 3 — Transformaciones
  df.withColumn()                  → crear/modificar columna
  F.when().otherwise()             → lógica condicional
  F.concat() / F.upper()           → funciones de string
  F.udf() / @udf                   → funciones Python custom

NOTEBOOK 4 — Agregaciones
  df.groupBy().agg()               → métricas por grupo
  df.groupBy().pivot()             → tabla dinámica
  Window + rank/avg/sum/lag/lead   → funciones de ventana

NOTEBOOK 5 — SQL y Joins
  createOrReplaceTempView()        → DataFrame → tabla SQL
  spark.sql()                      → ejecutar SQL
  df.join(df2, cond, how=...)      → unir DataFrames
  F.broadcast()                    → optimizar joins

NOTEBOOK 6 — Pipeline completo
  Combinación de todo lo anterior para responder preguntas de negocio
  .coalesce(1) → un solo archivo de salida
  spark.stop() → liberar recursos
```